1. Conexión a la base de datos
2. Exploración inicial
3. Limpieza y preparación
4. Métricas generales del negocio
5. Análisis por fuente
6. Evaluación de eficiencia (CAC, LTV, ROMI)
7. Conclusión ejecutiva

# Conectarse a la base de datos

### Copia el siguiente código para crear una conexión a la base de datos:

In [28]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


### La conexión se almacena en la variable engine. Puedes ejecutar una consulta SQL utilizando pandas:

In [29]:
query = """
SELECT 1;
"""
pd.io.sql.read_sql(query, con = engine)

,?column?
0,1


Con este código se hace la consulta y vemos que ya estamos conectados.

In [30]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
"""
pd.read_sql(query, engine)

,table_name
0,advertisment_costs
1,authors
2,books
3,check_avg
4,orders
5,publishers
6,ratings
7,reviews
8,visits


Con el código se obtiene los nombres de las columnas y para esto vamos a elegir tres tablas: visits, orders y advertisment_costs.

¿Por qué elegimos estas 3 tablas y sus columnas?

Porque en este proyecto el foco casi siempre es marketing/ventas: medir si la publicidad trae visitas y compras, y si vale la pena (ROI/ROMI).

Estas 3 tablas forman el triángulo de marketing:

	•Conversion Rate (visita, compra)

	•CAC (costo por comprador)

	•ROMI/ROI (rentabilidad)

In [31]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema='public'
AND table_name='visits'
ORDER BY ordinal_position;
"""
pd.read_sql(query, engine)

,column_name,data_type
0,id,integer
1,uid,character varying
2,device,character varying
3,endts,timestamp without time zone
4,sourceid,smallint
5,startts,timestamp without time zone


In [32]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema='public'
AND table_name='orders'
ORDER BY ordinal_position;
"""
pd.read_sql(query, engine)

,column_name,data_type
0,id,integer
1,buyts,timestamp without time zone
2,revenue,money
3,uid,character varying


In [33]:
query = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema='public'
AND table_name='advertisment_costs'
ORDER BY ordinal_position;
"""
pd.read_sql(query, engine)

,column_name,data_type
0,id,integer
1,sourceid,smallint
2,dt,timestamp without time zone
3,costs,money


En estas consultas de las tablas correspondientes con el código le preguntamos a PostgreSQL. Dime que columnas tienen las tablas "visits", "orders" y "advertisment_costs" y que tipo de datos tienen cada una

### Cargar datos a Dataframe

In [34]:
pd.read_sql("SELECT * FROM visits LIMIT 5;", engine)

,id,uid,device,endts,sourceid,startts
0,0,16879256277535980062,touch,2017-12-20 17:38:00,4,2017-12-20 17:20:00
1,1,104060357244891740,desktop,2018-02-19 17:21:00,2,2018-02-19 16:53:00
2,2,7459035603376831527,touch,2017-07-01 01:54:00,5,2017-07-01 01:54:00
3,3,16174680259334210214,desktop,2018-05-20 11:23:00,9,2018-05-20 10:59:00
4,4,9969694820036681168,desktop,2017-12-27 14:06:00,3,2017-12-27 14:06:00


In [35]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", engine)

,id,buyts,revenue,uid
0,1,2017-06-01 00:10:00,$17.00,10329302124590727494
1,2,2017-06-01 00:25:00,$0.55,11627257723692907447
2,3,2017-06-01 00:27:00,$0.37,17903680561304213844
3,4,2017-06-01 00:29:00,$0.55,16109239769442553005
4,5,2017-06-01 07:58:00,$0.37,14200605875248379450


In [36]:
pd.read_sql("SELECT * FROM advertisment_costs LIMIT 5;", engine)

,id,sourceid,dt,costs
0,1,1,2017-06-01,$75.20
1,2,1,2017-06-02,$62.25
2,3,1,2017-06-03,$36.53
3,4,1,2017-06-04,$55.00
4,5,1,2017-06-05,$57.08


In [37]:
# Vamos a cargar los datos en dataframes para su posterior análisis
visits = pd.read_sql("SELECT * FROM visits;", engine)
orders = pd.read_sql("SELECT * FROM orders;", engine)
costs  = pd.read_sql("SELECT * FROM advertisment_costs;", engine)

In [38]:
visits.info()
orders.info()
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 358532 entries, 0 to 358531
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype         
---  ------    --------------   -----         
 0   id        358532 non-null  int64         
 1   uid       358532 non-null  object        
 2   device    358532 non-null  object        
 3   endts     358532 non-null  datetime64[ns]
 4   sourceid  358532 non-null  int64         
 5   startts   358532 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(2), object(2)
memory usage: 16.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   id       50415 non-null  int64         
 1   buyts    50415 non-null  datetime64[ns]
 2   revenue  50415 non-null  object        
 3   uid      50415 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 1.5+ 

Por medio de .info() podemos ver que en la tabla orders la columna revenue y en la tabla de costs la columna costs la tenemos como objects y la tenemos que convertir a float para calcular las estadísticas correctas.

In [39]:
orders['revenue'] = (
    orders['revenue']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

costs['costs'] = (
    costs['costs']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

In [40]:
# Verificamos que los cambios se hayan realizado correctamente
orders.info()
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   id       50415 non-null  int64         
 1   buyts    50415 non-null  datetime64[ns]
 2   revenue  50415 non-null  float64       
 3   uid      50415 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 1.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2542 entries, 0 to 2541
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   id        2542 non-null   int64         
 1   sourceid  2542 non-null   int64         
 2   dt        2542 non-null   datetime64[ns]
 3   costs     2542 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 79.6 KB


### ¿Cuánto tráfico hubo?

In [41]:
# Total de visitas
total_visits = visits.shape[0]
total_visits

358532

### ¿Cuántos usuarios únicos?

In [42]:
unique_users = visits['uid'].nunique()
unique_users

228169

### ¿Cuántos compraron?

In [43]:
# Total de compras
total_orders = orders.shape[0]
print(total_orders)

# Compradores únicos
unique_buyers = orders['uid'].nunique()
print(unique_buyers)

50415
36523


### ¿Cuánto dinero entró?

In [44]:
total_revenue = orders['revenue'].sum()
total_revenue

252057.19999999998

### ¿Cuánto se gastó?

In [45]:
total_costs = costs['costs'].sum()
total_costs

329131.62

### ROMI(Rentabilidad)

In [46]:
# Calculemos la rentabilidad (ROMI general)
romi = (total_revenue - total_costs) / total_costs
romi

-0.2341750695360112

Como podemos ver el ROMI(rentabilidad) es negativa ya que hay mas gastos que ingresos.

ROMI = 23%

Eso significa que por cada $1 invertido, se recuperaron $0.77.

### CAC general (Costo por comprador)

In [47]:
cac = total_costs / unique_buyers
cac

9.011626098622786

### Ingreso promedio por comprador (LTV simple)

In [48]:
ltv = total_revenue / unique_buyers
ltv

6.9013279303452615

El análisis general muestra que el negocio no es rentable en términos globales. 
El ROMI general es negativo (-23%), lo que indica que los ingresos generados no compensan el gasto en publicidad.
Además, el costo de adquisición de cliente (CAC) es mayor que el ingreso promedio por cliente (LTV), lo que confirma una estructura de adquisición ineficiente.

Estamos pagando $9.01 para adquirir un cliente.

Ese cliente genera solo $6.90

## Análisis por fuente 

Primero necesitamos saber:

•¿Cuántos usuarios llegaron por cada sourceid?

•¿Cuántos compraron?

•¿Cuánto ingresaron?

•¿Cuánto costaron?

In [49]:
# Usuarios por fuente de tráfico
users_by_source = visits.groupby('sourceid')['uid'].nunique().reset_index()
users_by_source.columns = ['sourceid', 'unique_users']
print(users_by_source)

# Primera fuente de tráfico
first_source = (
    visits
    .sort_values('startts')
    .groupby('uid')
    .first()
    .reset_index()[['uid', 'sourceid']]
)

print(first_source.head())


   sourceid  unique_users
0         1         18999
1         2         26245
2         3         74756
3         4         83525
4         5         56974
5         6             6
6         7            34
7         9          9264
8        10          8067
                    uid  sourceid
0  10000171586432207426         3
1  10000344846682484395         3
2   1000036778515242839         3
3  10000460875579931334         4
4  10000558740593440652         4


### ¿Cuántos compradores trajo cada fuente de tráfico?

In [50]:
buyers = orders[['uid']].drop_duplicates()

# unimos compradores con su primera fuente de tráfico
buyers_by_source = (
    first_source
    .merge(buyers, on='uid', how='inner')
    .groupby('sourceid')['uid']
    .nunique()
    .reset_index(name='buyers')
)
buyers_by_source

,sourceid,buyers
0,1,2920
1,2,3497
2,3,10464
3,4,10280
4,5,6960
5,7,1
6,9,1074
7,10,1327


En la tabla podemos ver las diferentes fuentes y la cantidad de compradores únicos.

Fuentes más fuertes en volumen:

	•Source 3 10,464 compradores

	•Source 4  10,280 compradores

	•Source 5  6,960 compradores

Fuente muy débil:

	•Source 7 solo 1 comprador

Eso ya es una alerta.

Aqui podemos ver dos puntos de vista muy importantes, las fuentes 3 y 4 generan mucha compra pero no sabemos si salga carisima la publicidad y termine perdiendo dinero.

La fuente 6 y 8 no aparecen eso significa que no generan nada de conversión.

### Cuánto tuvo de ingreso por cada fuente?

In [51]:
revenue_by_user = (
    orders
    .groupby('uid')['revenue']
    .sum()
    .reset_index()
)

revenue_by_source = (
    first_source
    .merge(revenue_by_user, on='uid', how='inner')
    .groupby('sourceid')['revenue']
    .sum()
    .reset_index()
)

revenue_by_source

,sourceid,revenue
0,1,29876.00
1,2,46574.17
2,3,54391.15
3,4,56682.99
4,5,54407.71
5,7,1.22
6,9,5666.56
7,10,4457.40


### Análisis de fuentes con mayores ingresos

El análisis de ingresos por fuente muestra que las fuentes 3, 4 y 5 concentran la mayor generación de revenue, cada una superando los 54 mil en ingresos totales. La fuente 4 lidera ligeramente con aproximadamente 56.7 mil.

Esto indica que estas fuentes no solo generan tráfico, sino que atraen usuarios con alta probabilidad de compra y con un valor de cliente relevante.

La fuente 2 también presenta un desempeño sólido, con más de 46 mil en ingresos, posicionándose como una fuente estratégica dentro del mix de adquisición.

Sin embargo, un alto volumen de ingresos no implica necesariamente rentabilidad. Para evaluar la eficiencia real de estas fuentes, es necesario comparar estos ingresos contra los costos de adquisición asociados a cada fuente y calcular métricas como CAC y ROMI.

### ¿Cuántos compradores terminaron comprando?

In [52]:
summary_temp = buyers_by_source.merge(revenue_by_source, on='sourceid')
summary_temp['ltv'] = summary_temp['revenue'] / summary_temp['buyers']
summary_temp

,sourceid,buyers,revenue,ltv
0,1,2920,29876.00,10.231507
1,2,3497,46574.17,13.318321
2,3,10464,54391.15,5.197931
3,4,10280,56682.99,5.513910
4,5,6960,54407.71,7.817200
5,7,1,1.22,1.220000
6,9,1074,5666.56,5.276127
7,10,1327,4457.40,3.359005


### Análisis del ingreso promedio por comprador (LTV) por fuente

El cálculo del LTV (ingreso promedio por comprador) revela diferencias importantes en la calidad de los usuarios adquiridos por cada fuente.

Las fuentes 2 y 1 presentan los valores más altos de LTV, con aproximadamente 13.3 y 10.2 respectivamente. Esto indica que los usuarios provenientes de estas fuentes generan un mayor ingreso promedio por cliente, lo que sugiere una mayor calidad del tráfico.

Por otro lado, las fuentes 3 y 4, aunque generan un alto volumen de compradores, muestran un LTV considerablemente menor (alrededor de 5.2 y 5.5). Esto indica que atraen muchos clientes, pero con un menor valor individual.

La fuente 5 presenta un desempeño intermedio con un LTV cercano a 7.8, mientras que las fuentes 9 y 10 muestran valores bajos, especialmente la fuente 10 con aproximadamente 3.36.

La fuente 7 presenta un caso extremo con un único comprador y un LTV de 1.22, lo que sugiere un rendimiento marginal.

En términos estratégicos, no solo es relevante el volumen de compradores, sino también la calidad del cliente adquirido. Las fuentes con mayor LTV podrían ser más atractivas si sus costos de adquisición son competitivos.

### ¿Cuánto fué el costo por fuente?

In [53]:
costs_by_source = (
    costs
    .groupby('sourceid')['costs']
    .sum()
    .reset_index()
)

costs_by_source

,sourceid,costs
0,1,20833.27
1,2,42806.04
2,3,141321.63
3,4,61073.60
4,5,51757.10
5,9,5517.49
6,10,5822.49


### Análisis de costos por fuente

El análisis de los costos de adquisición muestra diferencias significativas entre las fuentes de tráfico.

La fuente 3 concentra el mayor gasto publicitario, con más de 141 mil en inversión, superando ampliamente a todas las demás fuentes. Esto indica una estrategia agresiva de adquisición en este canal.

Las fuentes 4 y 5 presentan niveles de inversión intermedios, con aproximadamente 61 mil y 51 mil respectivamente. Estas fuentes parecen formar parte del núcleo principal de inversión publicitaria.

La fuente 2 también representa una inversión considerable (42 mil), mientras que la fuente 1 muestra un nivel de gasto moderado (20 mil).

Por otro lado, las fuentes 9 y 10 presentan inversiones significativamente menores, lo que podría indicar pruebas de canal, menor prioridad estratégica o menor capacidad de escala.

Es importante destacar que un mayor nivel de inversión no garantiza mejores resultados. Para evaluar la eficiencia real de cada fuente, es necesario contrastar estos costos con los ingresos generados y calcular métricas como CAC y ROMI.

### Calcula el CAC y el ROMI por fuente

In [54]:
summary_final = (
    summary_temp
    .merge(costs_by_source, on='sourceid')
)

summary_final['cac'] = summary_final['costs'] / summary_final['buyers']
summary_final['romi'] = (summary_final['revenue'] - summary_final['costs']) / summary_final['costs']

summary_final.sort_values('romi', ascending=False)

,sourceid,buyers,revenue,ltv,costs,cac,romi
0,1,2920,29876.00,10.231507,20833.27,7.134682,0.434052
1,2,3497,46574.17,13.318321,42806.04,12.240789,0.088028
4,5,6960,54407.71,7.817200,51757.10,7.436365,0.051212
5,9,1074,5666.56,5.276127,5517.49,5.137328,0.027018
3,4,10280,56682.99,5.513910,61073.60,5.941012,-0.071890
6,10,1327,4457.40,3.359005,5822.49,4.387709,-0.234451
2,3,10464,54391.15,5.197931,141321.63,13.505507,-0.615125


### Análisis de eficiencia por fuente (CAC y ROMI)

El análisis final muestra diferencias claras en la eficiencia de las fuentes de adquisición.

La fuente 1 presenta el mejor desempeño, con un ROMI de aproximadamente 43%. Su CAC (7.13) es considerablemente menor que su LTV (10.23), lo que indica una fuente rentable y eficiente.

La fuente 2 también es rentable, aunque con un margen mucho menor (ROMI ~8.8%). A pesar de tener el LTV más alto (13.32), su costo de adquisición es elevado (12.24), lo que reduce significativamente su rentabilidad.

Las fuentes 5 y 9 muestran rentabilidad positiva, pero marginal (ROMI cercano a 5% y 2.7% respectivamente). Estas fuentes podrían optimizarse para mejorar su eficiencia.

Por otro lado, las fuentes 4, 10 y especialmente 3 presentan ROMI negativo. La fuente 3 destaca como la menos eficiente, con un ROMI de -61.5%, lo que indica que el gasto publicitario supera ampliamente los ingresos generados. Esta fuente está afectando significativamente la rentabilidad global del negocio.

En conclusión, el negocio en términos generales presenta un desempeño negativo debido principalmente a la fuerte inversión en fuentes ineficientes. Una estrategia de reasignación de presupuesto hacia las fuentes 1 y 2, junto con la reducción o optimización de la fuente 3, podría mejorar considerablemente la rentabilidad.

# Conclusiones finales

El análisis muestra que el negocio presenta una rentabilidad global negativa, principalmente debido a una alta inversión en fuentes de adquisición ineficientes.

La fuente 1 destaca como la más rentable, con un ROMI positivo significativo y un CAC inferior a su LTV. Esta fuente demuestra un modelo sostenible y debería considerarse prioritaria para aumentar la inversión.

La fuente 2 también es rentable, aunque con un margen reducido. Se recomienda optimizar su costo de adquisición para mejorar su eficiencia antes de escalarla.

Las fuentes 5 y 9 muestran rentabilidad marginal. Estas podrían mantenerse activas, pero con seguimiento constante y pruebas de optimización.

Por otro lado, la fuente 3 representa el principal problema financiero del negocio. Aunque genera alto volumen de compradores e ingresos, su costo es excesivo, lo que produce un ROMI altamente negativo. Se recomienda reducir significativamente su presupuesto o reevaluar la estrategia de adquisición en este canal.

Las fuentes 4 y 10 presentan rentabilidad negativa o cercana al punto de equilibrio. Se sugiere realizar pruebas de optimización antes de decidir su eliminación definitiva.

En conclusión, la estrategia óptima consiste en reasignar presupuesto desde las fuentes ineficientes hacia aquellas con mejor desempeño, priorizando la rentabilidad sobre el volumen de tráfico. Esto permitirá mejorar el ROMI global y construir un modelo de adquisición sostenible.